In [4]:
import numpy as np
import pandas as pd

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("gold_test_data_with_num_pings.csv")

# =========================
# COLUMN CHECKS
# =========================
required_cols = [
    "MMSI",
    "voyage_id",
    "TIME",
    "LAT",
    "LON",
    "SPEED",
    "COG",
    "HEADING",
    "dt",
    "num_pings"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# =========================
# PREP
# =========================
df["TIME"] = pd.to_datetime(df["TIME"], errors="coerce")
df = df.sort_values(["voyage_id", "TIME"]).reset_index(drop=True)

# New columns
heading_rad = np.deg2rad(df["HEADING"])
df["sin_heading"] = np.sin(heading_rad)
df["cos_heading"] = np.cos(heading_rad)

df["dr_pred_lat"] = np.nan
df["dr_pred_lon"] = np.nan
df["error_yds"] = np.nan

# optional replacement since row_id is gone
df["predicted_index"] = np.nan

# =========================
# CONSTANTS
# =========================
EARTH_RADIUS_M = 6371000.0
KNOT_TO_MPS = 0.514444
M_TO_YARDS = 1.09361

# =========================
# FUNCTIONS
# =========================
def dead_reckon(lat_deg, lon_deg, sog_knots, cog_deg, dt_sec):
    lat1 = np.radians(lat_deg)
    lon1 = np.radians(lon_deg)
    bearing = np.radians(cog_deg)

    distance_m = sog_knots * KNOT_TO_MPS * dt_sec
    angular_distance = distance_m / EARTH_RADIUS_M

    lat2 = np.arcsin(
        np.sin(lat1) * np.cos(angular_distance) +
        np.cos(lat1) * np.sin(angular_distance) * np.cos(bearing)
    )

    lon2 = lon1 + np.arctan2(
        np.sin(bearing) * np.sin(angular_distance) * np.cos(lat1),
        np.cos(angular_distance) - np.sin(lat1) * np.sin(lat2)
    )

    return np.degrees(lat2), np.degrees(lon2)


def haversine_m(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return EARTH_RADIUS_M * c

# =========================
# DEAD RECKONING WITHIN VOYAGE
# =========================
# current row i predicts next row i+1
# only if both are in the same voyage
# dt used is next row's dt, which represents elapsed time from i -> i+1

next_voyage_id = df["voyage_id"].shift(-1)
next_lat = df["LAT"].shift(-1)
next_lon = df["LON"].shift(-1)
next_dt = df["dt"].shift(-1)

# since row_id is gone, use next dataframe index instead
next_index = pd.Series(df.index, index=df.index).shift(-1)

valid_mask = (
    next_voyage_id.notna() &
    (df["voyage_id"] == next_voyage_id) &
    next_lat.notna() &
    next_lon.notna() &
    next_dt.notna() &
    (next_dt > 0) &
    df["LAT"].notna() &
    df["LON"].notna() &
    df["SPEED"].notna() &
    df["COG"].notna()
)

valid_idx = df.index[valid_mask]

pred_lats = []
pred_lons = []
errors_yds = []
predicted_indices = []

for i in valid_idx:
    pred_lat, pred_lon = dead_reckon(
        lat_deg=df.at[i, "LAT"],
        lon_deg=df.at[i, "LON"],
        sog_knots=df.at[i, "SPEED"],
        cog_deg=df.at[i, "COG"],
        dt_sec=next_dt.loc[i]
    )

    actual_next_lat = next_lat.loc[i]
    actual_next_lon = next_lon.loc[i]

    error_m = haversine_m(
        actual_next_lat,
        actual_next_lon,
        pred_lat,
        pred_lon
    )

    pred_lats.append(pred_lat)
    pred_lons.append(pred_lon)
    errors_yds.append(error_m * M_TO_YARDS)
    predicted_indices.append(next_index.loc[i])

df.loc[valid_idx, "dr_pred_lat"] = pred_lats
df.loc[valid_idx, "dr_pred_lon"] = pred_lons
df.loc[valid_idx, "error_yds"] = errors_yds
df.loc[valid_idx, "predicted_index"] = predicted_indices

# =========================
# OPTIONAL: SAVE
# =========================
#df.to_csv("gold_test_data_with_dr.csv", index=False)

# =========================
# QUICK CHECK
# =========================
display_cols = [
    "voyage_id",
    "predicted_index",
    "TIME",
    "LAT",
    "LON",
    "SPEED",
    "COG",
    "HEADING",
    "dt",
    "num_pings",
    "sin_heading",
    "cos_heading",
    "dr_pred_lat",
    "dr_pred_lon",
    "error_yds"
]

print(df[display_cols].head(15))
print("\nSaved as: gold_test_data_with_dr.csv")
print(f"Rows with DR predictions: {df['dr_pred_lat'].notna().sum()}")
print(f"Mean DR error (yds): {df['error_yds'].mean():.2f}")
print(f"Median DR error (yds): {df['error_yds'].median():.2f}")

worst_10 = df.nlargest(76, "error_yds")

print(worst_10[[
    "voyage_id", "predicted_index", "TIME",
    "LAT", "LON",
    "SPEED", "COG", "HEADING", "dt",
    "dr_pred_lat", "dr_pred_lon", "error_yds"
]].to_string())

ValueError: Missing required columns: ['LAT', 'LON', 'SPEED', 'COG', 'HEADING', 'dt']

In [3]:
import numpy as np
import pandas as pd

# =========================
# LOAD CSV
# =========================
df = pd.read_csv("gold_test_data_with_dr.csv")

# =========================
# CLEAN ERROR COLUMN
# =========================
errors = df["error_yds"].dropna()

if len(errors) == 0:
    raise ValueError("No non-null values found in error_yds")

# =========================
# METRICS
# =========================
rmse_yds = np.sqrt(np.mean(errors ** 2))
mae_yds = np.mean(errors)
median_yds = np.median(errors)
p95_yds = np.percentile(errors, 95)

q1_yds = np.percentile(errors, 25)
q3_yds = np.percentile(errors, 75)
iqr_yds = q3_yds - q1_yds

lower_bound_yds = q1_yds - 1.5 * iqr_yds
upper_bound_yds = q3_yds + 1.5 * iqr_yds

outliers = errors[(errors < lower_bound_yds) | (errors > upper_bound_yds)]
outlier_count = len(outliers)
outlier_percent = 100 * outlier_count / len(errors)

# =========================
# OUTPUT
# =========================
print(f"rmse_yds: {rmse_yds:.2f}")
print(f"mae_yds: {mae_yds:.2f}")
print(f"median_yds: {median_yds:.2f}")
print(f"p95_yds: {p95_yds:.2f}")
print(f"q1_yds: {q1_yds:.2f}")
print(f"q3_yds: {q3_yds:.2f}")
print(f"iqr_yds: {iqr_yds:.2f}")
print(f"lower_bound_yds: {lower_bound_yds:.2f}")
print(f"upper_bound_yds: {upper_bound_yds:.2f}")
print(f"outlier_count: {outlier_count}")
print(f"outlier_percent: {outlier_percent:.2f}")

rmse_yds: 1701.45
mae_yds: 613.80
median_yds: 320.85
p95_yds: 1839.71
q1_yds: 89.69
q3_yds: 845.49
iqr_yds: 755.80
lower_bound_yds: -1044.01
upper_bound_yds: 1979.19
outlier_count: 58
outlier_percent: 2.90


In [6]:
import numpy as np
import pandas as pd

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("mid_datav4.csv")

# =========================
# RENAME COLUMNS IF NEEDED
# =========================
rename_map = {
    "LAT_AVG": "LAT",
    "LON_AVG": "LON",
    "SPEED_KNOTS": "SPEED",
    "COG_DEG": "COG",
    "HEADING_DEG": "HEADING",
    "time_diff": "dt"
}
df = df.rename(columns=rename_map)

# =========================
# COLUMN CHECKS
# =========================
required_cols = [
    "MMSI",
    "voyage_id",
    "TIME",
    "LAT",
    "LON",
    "SPEED",
    "COG",
    "HEADING",
    "dt",
    "num_pings"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# =========================
# PREP
# =========================
df["TIME"] = pd.to_datetime(df["TIME"], errors="coerce")
df = df.sort_values(["voyage_id", "TIME"]).reset_index(drop=True)

heading_rad = np.deg2rad(df["HEADING"])
df["sin_heading"] = np.sin(heading_rad)
df["cos_heading"] = np.cos(heading_rad)

df["dr_pred_lat"] = np.nan
df["dr_pred_lon"] = np.nan
df["error_yds"] = np.nan
df["predicted_index"] = np.nan

# =========================
# CONSTANTS
# =========================
EARTH_RADIUS_M = 6371000.0
KNOT_TO_MPS = 0.514444
M_TO_YARDS = 1.09361

# =========================
# FUNCTIONS
# =========================
def dead_reckon(lat_deg, lon_deg, sog_knots, cog_deg, dt_sec):
    lat1 = np.radians(lat_deg)
    lon1 = np.radians(lon_deg)
    bearing = np.radians(cog_deg)

    distance_m = sog_knots * KNOT_TO_MPS * dt_sec
    angular_distance = distance_m / EARTH_RADIUS_M

    lat2 = np.arcsin(
        np.sin(lat1) * np.cos(angular_distance) +
        np.cos(lat1) * np.sin(angular_distance) * np.cos(bearing)
    )

    lon2 = lon1 + np.arctan2(
        np.sin(bearing) * np.sin(angular_distance) * np.cos(lat1),
        np.cos(angular_distance) - np.sin(lat1) * np.sin(lat2)
    )

    return np.degrees(lat2), np.degrees(lon2)


def haversine_m(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return EARTH_RADIUS_M * c

# =========================
# DEAD RECKONING WITHIN VOYAGE
# =========================
next_voyage_id = df["voyage_id"].shift(-1)
next_lat = df["LAT"].shift(-1)
next_lon = df["LON"].shift(-1)
next_dt = df["dt"].shift(-1)
next_index = pd.Series(df.index, index=df.index).shift(-1)

valid_mask = (
    next_voyage_id.notna() &
    (df["voyage_id"] == next_voyage_id) &
    next_lat.notna() &
    next_lon.notna() &
    next_dt.notna() &
    (next_dt > 0) &
    df["LAT"].notna() &
    df["LON"].notna() &
    df["SPEED"].notna() &
    df["COG"].notna()
)

valid_idx = df.index[valid_mask]

pred_lats = []
pred_lons = []
errors_yds = []
predicted_indices = []

for i in valid_idx:
    pred_lat, pred_lon = dead_reckon(
        lat_deg=df.at[i, "LAT"],
        lon_deg=df.at[i, "LON"],
        sog_knots=df.at[i, "SPEED"],
        cog_deg=df.at[i, "COG"],
        dt_sec=next_dt.loc[i]
    )

    actual_next_lat = next_lat.loc[i]
    actual_next_lon = next_lon.loc[i]

    error_m = haversine_m(
        actual_next_lat,
        actual_next_lon,
        pred_lat,
        pred_lon
    )

    pred_lats.append(pred_lat)
    pred_lons.append(pred_lon)
    errors_yds.append(error_m * M_TO_YARDS)
    predicted_indices.append(next_index.loc[i])

df.loc[valid_idx, "dr_pred_lat"] = pred_lats
df.loc[valid_idx, "dr_pred_lon"] = pred_lons
df.loc[valid_idx, "error_yds"] = errors_yds
df.loc[valid_idx, "predicted_index"] = predicted_indices

# =========================
# QUICK CHECK
# =========================
display_cols = [
    "voyage_id",
    "predicted_index",
    "TIME",
    "LAT",
    "LON",
    "SPEED",
    "COG",
    "HEADING",
    "dt",
    "num_pings",
    "sin_heading",
    "cos_heading",
    "dr_pred_lat",
    "dr_pred_lon",
    "error_yds"
]

print(df[display_cols].head(15))

mse_yds = np.mean(df["error_yds"].dropna() ** 2)
rmse_yds = np.sqrt(mse_yds)
mean_yds = df["error_yds"].mean()
median_yds = df["error_yds"].median()

print(f"\nRows with DR predictions: {df['dr_pred_lat'].notna().sum()}")
print(f"MSE DR error (yds^2): {mse_yds:.2f}")
print(f"RMSE DR error (yds): {rmse_yds:.2f}")
print(f"Mean DR error (yds): {mean_yds:.2f}")
print(f"Median DR error (yds): {median_yds:.2f}")

worst_10 = df.nlargest(76, "error_yds")

print(worst_10[[
    "voyage_id", "predicted_index", "TIME",
    "LAT", "LON",
    "SPEED", "COG", "HEADING", "dt",
    "dr_pred_lat", "dr_pred_lon", "error_yds"
]].to_string())

       voyage_id  predicted_index                TIME        LAT        LON  \
0   10_209425000              1.0 2024-03-03 10:55:00  22.348651 -77.748852   
1   10_209425000              2.0 2024-03-03 11:00:00  22.334810 -77.726559   
2   10_209425000              3.0 2024-03-03 11:05:00  22.327325 -77.714589   
3   10_209425000              4.0 2024-03-03 11:10:00  22.316326 -77.696899   
4   10_209425000              5.0 2024-03-03 11:15:00  22.305286 -77.681877   
5   10_209425000              6.0 2024-03-03 11:20:00  22.292025 -77.664940   
6   10_209425000              7.0 2024-03-03 11:25:00  22.278540 -77.648339   
7   10_209425000              8.0 2024-03-03 11:30:00  22.263906 -77.630150   
8   10_209425000              9.0 2024-03-03 11:35:00  22.249001 -77.611604   
9   10_209425000             10.0 2024-03-03 11:40:00  22.234013 -77.592872   
10  10_209425000             11.0 2024-03-03 11:45:00  22.218496 -77.574205   
11  10_209425000             12.0 2024-03-03 11:50:0